In [7]:
# ============================================================================
# 0. ENVIRONMENT SETUP
# ============================================================================
import numpy as np
import pandas as pd
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, LSTM, Dropout, Bidirectional, Input,
    LeakyReLU, BatchNormalization, RepeatVector, TimeDistributed
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU Enabled: {len(gpus)}")
else:
    print("⚠️ Running on CPU (training will be slower)")


✅ GPU Enabled: 1


In [8]:

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
# Paths — adjust if your project root is different
# Paths — adjust if your project root is different
try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined in Jupyter/Colab
    PROJECT_ROOT = os.getcwd()

# If running from the project root directly:
if not os.path.exists(os.path.join(PROJECT_ROOT, 'models')):
    # Try one level up if inside a subdirectory (e.g. GAN_MODEL)
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
    if not os.path.exists(os.path.join(PROJECT_ROOT, 'models')):
        # Fallback to current working directory if all else fails
        PROJECT_ROOT = os.getcwd()

# ----------------------------------------------------------------------------
# COLAB HELPER: Asset Verification & Google Drive Mount
# ----------------------------------------------------------------------------
def ensure_colab_assets():
    """Checks for files. If missing, suggests Drive mount or Drag-and-Drop."""
    import sys
    if 'google.colab' not in sys.modules:
        return

    # Expected artifacts relative to PROJECT_ROOT
    required = [
        os.path.join('models', 'saved_models', 'gan_generator_ohlcv.h5'),
        os.path.join('models', 'saved_models', 'lstm_model.h5'),
        os.path.join('models', 'scalers', 'lstm_model_scaler.pkl'),
        os.path.join('data', 'raw', 'ogdc_data_updated.csv')
    ]
    
    missing = []
    for rel_path in required:
        full_path = os.path.join(PROJECT_ROOT, rel_path)
        if not os.path.exists(full_path):
            missing.append(rel_path)
    
    if not missing:
        print("✅ All assets found.")
        return

    print("\n" + "!"*80)
    print(f"⚠️ MISSING {len(missing)} FILES IN COLAB RUNTIME:")
    for rel in missing:
        print(f"   - {rel}")
    print("!"*80)
    print("\nOPTION 1: GOOGLE DRIVE (Recommended)")
    print("   1. Upload your 'models' and 'data' folders to your Google Drive root.")
    print("   2. Run the cell below to mount Drive.")
    print("   3. The script will link '/content/drive/MyDrive/...' to the expected paths.")
    
    print("\nOPTION 2: DRAG & DROP")
    print("   1. Open the file explorer in the left sidebar.")
    print("   2. Drag your local 'models' and 'data' folders into the runtime.")
    print("!"*80 + "\n")

    # Attempt to mount drive automatically
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            print("Mounting Google Drive...")
            drive.mount('/content/drive')
        
        # Check if files exist in Drive and symlink them
        drive_root = '/content/drive/MyDrive/GAN PSX Marketforecasting'
        # If user just dumped them in root:
        if not os.path.exists(drive_root):
            drive_root = '/content/drive/MyDrive'
            
        print(f"Checking Drive path: {drive_root}")
        found_in_drive = False
        for rel in missing:
            drive_path = os.path.join(drive_root, rel)
            if os.path.exists(drive_path):
                # Create local directory structure
                local_dest = os.path.join(PROJECT_ROOT, rel)
                os.makedirs(os.path.dirname(local_dest), exist_ok=True)
                # Copy or Symlink
                if not os.path.exists(local_dest):
                    import shutil
                    shutil.copy(drive_path, local_dest)
                    print(f"✅ Restored from Drive: {rel}")
                    found_in_drive = True
        
        if found_in_drive:
             print("Assets restored from Google Drive! proceeding...")
             return

    except Exception as e:
        print(f"Drive mount verification failed/skipped: {e}")

    print("\n❌ STILL MISSING FILES. Please upload them manually to continue.")
    # Stop execution if files are strictly required
    # raise FileNotFoundError("Missing critical training files.")

# Run the check before defining strict paths
ensure_colab_assets()
# ----------------------------------------------------------------------------

GAN_WEIGHTS_PATH   = os.path.join(PROJECT_ROOT, 'models', 'saved_models', 'gan_generator_ohlcv.h5')
LSTM_WEIGHTS_PATH  = os.path.join(PROJECT_ROOT, 'models', 'saved_models', 'lstm_model.h5')
CALIBRATED_PATH    = os.path.join(PROJECT_ROOT, 'models', 'saved_models', 'lstm_model_calibrated.h5')
SCALER_PATH        = os.path.join(PROJECT_ROOT, 'models', 'scalers', 'lstm_model_scaler.pkl')
DATA_PATH          = os.path.join(PROJECT_ROOT, 'data', 'raw', 'ogdc_data_updated.csv')

# Hyperparameters
LOOKBACK            = 60       # Sequence length for LSTM
LATENT_DIM          = 100      # GAN noise dimension
GAN_SEQ_LEN         = 30       # GAN output sequence length
GAN_NUM_FEATURES    = 5        # OHLCV
CLOSE_COL_IDX       = 3        # Column index for Close in GAN output

N_SYNTHETIC_DAYS    = 5000     # How many synthetic closing prices to generate
STAGE_A_EPOCHS      = 30       # Pre-training epochs on synthetic data
STAGE_B_EPOCHS      = 50       # Fine-tuning epochs on real data
BATCH_SIZE          = 32
LEARNING_RATE_A     = 0.001    # Stage A learning rate (broad generalization)
LEARNING_RATE_B     = 0.0003   # Stage B learning rate (fine-tuning — lower)

print("\n" + "="*70)
print("  NEXUS AI — GAN→LSTM Augmented Training")
print("="*70)
print(f"  GAN Weights:      {GAN_WEIGHTS_PATH}")
print(f"  LSTM Weights:     {LSTM_WEIGHTS_PATH}")
print(f"  Output (calib):   {CALIBRATED_PATH}")
print(f"  Real Data:        {DATA_PATH}")
print(f"  Synthetic Days:   {N_SYNTHETIC_DAYS}")
print(f"  Stage A Epochs:   {STAGE_A_EPOCHS}")
print(f"  Stage B Epochs:   {STAGE_B_EPOCHS}")
print("="*70 + "\n")


Mounted at /content/drive
Checking Drive path: /content/drive/MyDrive
✅ Restored from Drive: models/saved_models/gan_generator_ohlcv.h5
✅ Restored from Drive: models/saved_models/lstm_model.h5
✅ Restored from Drive: models/scalers/lstm_model_scaler.pkl
✅ Restored from Drive: data/raw/ogdc_data_updated.csv
Assets restored from Google Drive! proceeding...

  NEXUS AI — GAN→LSTM Augmented Training
  GAN Weights:      /content/models/saved_models/gan_generator_ohlcv.h5
  LSTM Weights:     /content/models/saved_models/lstm_model.h5
  Output (calib):   /content/models/saved_models/lstm_model_calibrated.h5
  Real Data:        /content/data/raw/ogdc_data_updated.csv
  Synthetic Days:   5000
  Stage A Epochs:   30
  Stage B Epochs:   50



In [9]:

# ============================================================================
# 2. BUILD ARCHITECTURES
# ============================================================================

def build_gan_generator(latent_dim=LATENT_DIM, seq_len=GAN_SEQ_LEN, num_features=GAN_NUM_FEATURES):
    """Exact same architecture as models_engine.py"""
    model = Sequential(name="Generator")
    model.add(Input(shape=(latent_dim,)))
    model.add(Dense(128))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(RepeatVector(seq_len))
    model.add(LSTM(128, return_sequences=True))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LSTM(64, return_sequences=True))
    model.add(BatchNormalization(momentum=0.8))
    model.add(TimeDistributed(Dense(num_features, activation='sigmoid')))
    return model


def build_lstm_model(lookback=LOOKBACK):
    """Exact same architecture as models_engine.py / LSTM_Train.ipynb"""
    model = Sequential(name="LSTM_Forecaster")
    model.add(Input(shape=(lookback, 1)))
    model.add(Bidirectional(LSTM(100, return_sequences=True)))
    model.add(Dropout(0.3))
    model.add(LSTM(100, return_sequences=False))
    model.add(Dropout(0.3))
    model.add(Dense(25, activation='relu'))
    model.add(Dense(1))
    return model



In [10]:

# ============================================================================
# 3. STAGE A — GENERATE SYNTHETIC DATA VIA GAN
# ============================================================================
print("━"*70)
print("  STAGE A: Generating Synthetic Data via GAN")
print("━"*70)

# 3a. Load GAN Generator
gan = build_gan_generator()
if os.path.exists(GAN_WEIGHTS_PATH):
    gan.load_weights(GAN_WEIGHTS_PATH)
    print(f"✅ GAN Generator weights loaded from {GAN_WEIGHTS_PATH}")
else:
    raise FileNotFoundError(
        f"❌ GAN weights not found at {GAN_WEIGHTS_PATH}. "
        "Train the GAN first using GAN_MODEL/GAN.ipynb"
    )

# 3b. GAN scaler (same dummy calibration as models_engine.py)
gan_scaler = MinMaxScaler(feature_range=(0, 1))
gan_scaler.fit(np.array([[100]*GAN_NUM_FEATURES, [300]*GAN_NUM_FEATURES]))

# 3c. Generate synthetic Close prices
n_batches = N_SYNTHETIC_DAYS // GAN_SEQ_LEN + 1
synthetic_close = []

print(f"  Generating {n_batches} batches × {GAN_SEQ_LEN} days...")
for i in range(n_batches):
    noise = np.random.normal(0, 1, (1, LATENT_DIM))
    gen_seq = gan.predict(noise, verbose=0)  # (1, 30, 5)
    # Inverse-transform to real price scale
    real_scale = gan_scaler.inverse_transform(gen_seq[0])  # (30, 5)
    close_prices = real_scale[:, CLOSE_COL_IDX]  # (30,)
    synthetic_close.extend(close_prices.tolist())

synthetic_close = np.array(synthetic_close[:N_SYNTHETIC_DAYS])
print(f"✅ Generated {len(synthetic_close)} synthetic Close prices")
print(f"   Range: [{synthetic_close.min():.2f}, {synthetic_close.max():.2f}]")
print(f"   Mean:  {synthetic_close.mean():.2f}")

# 3d. Also load real data for combined scaler
df_real = pd.read_csv(DATA_PATH)
real_close = df_real['Close'].values.astype(float)
print(f"\n✅ Real OGDC data loaded: {len(real_close)} days")
print(f"   Range: [{real_close.min():.2f}, {real_close.max():.2f}]")

# 3e. Fit a COMBINED scaler on both synthetic + real data
all_prices = np.concatenate([synthetic_close, real_close]).reshape(-1, 1)
combined_scaler = MinMaxScaler(feature_range=(0, 1))
combined_scaler.fit(all_prices)
print(f"\n✅ Combined scaler fit on {len(all_prices)} data points")
print(f"   Data range: [{all_prices.min():.2f}, {all_prices.max():.2f}]")


# ============================================================================
# HELPER: Create LSTM sequences
# ============================================================================
def create_sequences(data, lookback=LOOKBACK):
    """Create (X, y) pairs for LSTM training.
    X: (n_samples, lookback, 1)
    y: (n_samples,)
    """
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X).reshape(-1, lookback, 1), np.array(y)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STAGE A: Generating Synthetic Data via GAN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ GAN Generator weights loaded from /content/models/saved_models/gan_generator_ohlcv.h5
  Generating 167 batches × 30 days...
✅ Generated 5000 synthetic Close prices
   Range: [100.11, 299.64]
   Mean:  213.26

✅ Real OGDC data loaded: 1223 days
   Range: [69.77, 333.90]

✅ Combined scaler fit on 6223 data points
   Data range: [69.77, 333.90]


In [11]:

# ============================================================================
# 4. STAGE A — PRE-TRAIN LSTM ON SYNTHETIC DATA
# ============================================================================
print("\n" + "━"*70)
print("  STAGE A: Pre-Training LSTM on Synthetic Data")
print("━"*70)

# 4a. Scale synthetic data
synthetic_scaled = combined_scaler.transform(synthetic_close.reshape(-1, 1))
X_synth, y_synth = create_sequences(synthetic_scaled, LOOKBACK)
print(f"  Synthetic training set: X={X_synth.shape}, y={y_synth.shape}")

# 4b. Build fresh LSTM
lstm = build_lstm_model(LOOKBACK)
lstm.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE_A),
    loss='mean_squared_error',
    metrics=['mae']
)
lstm.summary()

# 4c. Train on synthetic data
print(f"\n🔄 Training on {len(X_synth)} synthetic sequences for {STAGE_A_EPOCHS} epochs...")
history_a = lstm.fit(
    X_synth, y_synth,
    epochs=STAGE_A_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ],
    verbose=1
)

final_loss_a = history_a.history['val_loss'][-1]
print(f"\n✅ Stage A complete — Val Loss: {final_loss_a:.6f}")
print(f"   LSTM has now seen {len(X_synth)} synthetic patterns")




━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STAGE A: Pre-Training LSTM on Synthetic Data
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Synthetic training set: X=(4940, 60, 1), y=(4940,)


Model: "LSTM_Forecaster"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 60, 200)        │        81,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 200)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 100)            │       120,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 25)             │         2,525 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 204,551 (799.03 KB)

 Trainable params: 204,551 (799.03 KB)

 Non-trainable params: 0 (0.00 B)


🔄 Training on 4940 synthetic sequences for 30 epochs...
Epoch 1/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - loss: 0.1102 - mae: 0.2776 - val_loss: 0.0326 - val_mae: 0.1416 - learning_rate: 0.0010
Epoch 2/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0341 - mae: 0.1417 - val_loss: 0.0222 - val_mae: 0.1103 - learning_rate: 0.0010
Epoch 3/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0234 - mae: 0.1115 - val_loss: 0.0122 - val_mae: 0.0715 - learning_rate: 0.0010
Epoch 4/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0182 - mae: 0.0955 - val_loss: 0.0090 - val_mae: 0.0609 - learning_rate: 0.0010
Epoch 5/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0156 - mae: 0.0845 - val_loss: 0.0156 - val_mae: 0.0630 - learning_rate: 0.0010
Epoch 6/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0157 - mae: 0.0831 - val_loss: 0.0125 - val_mae: 0.0735 - learning_rate: 0.0010
Epoch 7/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.0137 - mae: 0.0788 - v

In [12]:

# ============================================================================
# 5. STAGE B — FINE-TUNE ON REAL OGDC DATA
# ============================================================================
print("\n" + "━"*70)
print("  STAGE B: Fine-Tuning on Real OGDC Data")
print("━"*70)

# 5a. Scale real data using the COMBINED scaler
real_scaled = combined_scaler.transform(real_close.reshape(-1, 1))
X_real, y_real = create_sequences(real_scaled, LOOKBACK)
print(f"  Real training set: X={X_real.shape}, y={y_real.shape}")

# 5b. Re-compile with LOWER learning rate for fine-tuning
lstm.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE_B),
    loss='mean_squared_error',
    metrics=['mae']
)

# 5c. Fine-tune on real data
print(f"\n🔄 Fine-tuning on {len(X_real)} real sequences for {STAGE_B_EPOCHS} epochs...")
history_b = lstm.fit(
    X_real, y_real,
    epochs=STAGE_B_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ],
    verbose=1
)

final_loss_b = history_b.history['val_loss'][-1]
print(f"\n✅ Stage B complete — Val Loss: {final_loss_b:.6f}")




━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STAGE B: Fine-Tuning on Real OGDC Data
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Real training set: X=(1163, 60, 1), y=(1163,)

🔄 Fine-tuning on 1163 real sequences for 50 epochs...
Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.1898 - mae: 0.3554 - val_loss: 0.1012 - val_mae: 0.2898 - learning_rate: 3.0000e-04
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0132 - mae: 0.0890 - val_loss: 0.0531 - val_mae: 0.2001 - learning_rate: 3.0000e-04
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0092 - mae: 0.0762 - val_loss: 0.0436 - val_mae: 0.1793 - learning_rate: 3.0000e-04
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0060 - mae: 0.0606 - val_loss: 0.0377 - val_mae: 0.1653 - learning_rate: 3.0000e-04
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0045 - mae: 0.0521 - val_loss: 0.0318 - val_mae: 0.1510 - learning_rate:

In [13]:

# ============================================================================
# 6. POST-TRAINING VALIDATION
# ============================================================================
print("\n" + "━"*70)
print("  POST-TRAINING VALIDATION")
print("━"*70)

# 6a. Predict on last 60 days of real data
last_60 = real_scaled[-LOOKBACK:]
X_test = last_60.reshape(1, LOOKBACK, 1)
pred_scaled = lstm.predict(X_test, verbose=0)
predicted_price = combined_scaler.inverse_transform(pred_scaled)[0][0]
actual_last = real_close[-1]

print(f"\n  Latest actual price:   Rs. {actual_last:.2f}")
print(f"  Calibrated prediction: Rs. {predicted_price:.2f}")
print(f"  Error:                 Rs. {abs(predicted_price - actual_last):.2f} "
      f"({abs(predicted_price - actual_last)/actual_last*100:.2f}%)")

# 6b. Directional accuracy on validation set
split = int(len(X_real) * 0.85)
X_val = X_real[split:]
y_val = y_real[split:]
preds_val = lstm.predict(X_val, verbose=0).flatten()

# Check directional accuracy
if len(y_val) > 1:
    actual_directions = np.diff(y_val) > 0
    pred_directions = np.diff(preds_val) > 0
    directional_accuracy = np.mean(actual_directions == pred_directions) * 100
    print(f"  Directional Accuracy:  {directional_accuracy:.1f}%")
else:
    directional_accuracy = 0
    print("  ⚠️ Not enough validation data for directional accuracy")




━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  POST-TRAINING VALIDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Latest actual price:   Rs. 321.18
  Calibrated prediction: Rs. 246.93
  Error:                 Rs. 74.25 (23.12%)
  Directional Accuracy:  49.4%


In [14]:

# ============================================================================
# 7. SAVE CALIBRATED MODEL + SCALER
# ============================================================================
print("\n" + "━"*70)
print("  SAVING CALIBRATED ARTIFACTS")
print("━"*70)

# 7a. Save calibrated model weights
os.makedirs(os.path.dirname(CALIBRATED_PATH), exist_ok=True)
lstm.save(CALIBRATED_PATH)
print(f"✅ Calibrated model saved: {CALIBRATED_PATH}")

# 7b. Also overwrite the original lstm_model.h5 for seamless integration
lstm.save(LSTM_WEIGHTS_PATH)
print(f"✅ Original model updated: {LSTM_WEIGHTS_PATH}")

# 7c. Save the combined scaler
os.makedirs(os.path.dirname(SCALER_PATH), exist_ok=True)
joblib.dump(combined_scaler, SCALER_PATH)
print(f"✅ Combined scaler saved: {SCALER_PATH}")




━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SAVING CALIBRATED ARTIFACTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Calibrated model saved: /content/models/saved_models/lstm_model_calibrated.h5
✅ Original model updated: /content/models/saved_models/lstm_model.h5
✅ Combined scaler saved: /content/models/scalers/lstm_model_scaler.pkl


In [15]:

# ============================================================================
# 8. TRAINING SUMMARY
# ============================================================================
print("\n" + "="*70)
print("  ✅ GAN→LSTM AUGMENTED TRAINING COMPLETE")
print("="*70)
print(f"""
  Stage A (Synthetic):
    • {len(X_synth)} samples, {STAGE_A_EPOCHS} epochs max
    • Final val_loss: {final_loss_a:.6f}
    
  Stage B (Real OGDC):
    • {len(X_real)} samples, {STAGE_B_EPOCHS} epochs max
    • Final val_loss: {final_loss_b:.6f}
    
  Validation:
    • Latest prediction: Rs. {predicted_price:.2f}
    • Directional accuracy: {directional_accuracy:.1f}%
    
  Saved Files:
    • {CALIBRATED_PATH}
    • {LSTM_WEIGHTS_PATH}
    • {SCALER_PATH}
    
  Next Steps:
    1. Run 'python src/backtest.py' to verify improved accuracy
    2. git add models/ && git commit -m "feat: GAN-augmented LSTM weights (curriculum learning)"
    3. Restart Flask server to load calibrated weights
""")



  ✅ GAN→LSTM AUGMENTED TRAINING COMPLETE

  Stage A (Synthetic):
    • 4940 samples, 30 epochs max
    • Final val_loss: 0.001746
    
  Stage B (Real OGDC):
    • 1163 samples, 50 epochs max
    • Final val_loss: 0.027780
    
  Validation:
    • Latest prediction: Rs. 246.93
    • Directional accuracy: 49.4%
    
  Saved Files:
    • /content/models/saved_models/lstm_model_calibrated.h5
    • /content/models/saved_models/lstm_model.h5
    • /content/models/scalers/lstm_model_scaler.pkl
    
  Next Steps:
    1. Run 'python src/backtest.py' to verify improved accuracy
    2. git add models/ && git commit -m "feat: GAN-augmented LSTM weights (curriculum learning)"
    3. Restart Flask server to load calibrated weights



In [16]:

# ============================================================================
# 9. BACKUP TO GOOGLE DRIVE
# ============================================================================
if os.path.exists('/content/drive/MyDrive'):
    print("\n" + "━"*70)
    print("  BACKING UP TO GOOGLE DRIVE")
    print("━"*70)
    
    # Try to find where the 'models' folder is located in Drive
    drive_roots = [
        '/content/drive/MyDrive/GAN PSX Marketforecasting',
        '/content/drive/MyDrive'
    ]
    
    target_root = None
    for r in drive_roots:
        if os.path.exists(os.path.join(r, 'models')):
            target_root = r
            break
            
    if target_root:
        drive_model_dir = os.path.join(target_root, 'models', 'saved_models')
        drive_scaler_dir = os.path.join(target_root, 'models', 'scalers')
        
        os.makedirs(drive_model_dir, exist_ok=True)
        os.makedirs(drive_scaler_dir, exist_ok=True)
        
        import shutil
        try:
            shutil.copy(CALIBRATED_PATH, os.path.join(drive_model_dir, 'lstm_model_calibrated.h5'))
            print(f"✅ Copied calibrated weights to Drive: .../saved_models/lstm_model_calibrated.h5")
            
            shutil.copy(LSTM_WEIGHTS_PATH, os.path.join(drive_model_dir, 'lstm_model.h5'))
            print(f"✅ Copied updated original weights to Drive")
            
            shutil.copy(SCALER_PATH, os.path.join(drive_scaler_dir, 'lstm_model_scaler.pkl'))
            print(f"✅ Copied combined scaler to Drive")
        except Exception as e:
            print(f"⚠️ Backup failed: {e}")
    else:
        print("⚠️ Could not locate 'models' folder in Drive to save back to.")



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BACKING UP TO GOOGLE DRIVE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Copied calibrated weights to Drive: .../saved_models/lstm_model_calibrated.h5
✅ Copied updated original weights to Drive
✅ Copied combined scaler to Drive
